## 0. Title, learning objectives, roadmap (3 min)

- **Course:** Scientific Programming for Physics (Oct 27, 2026)
- **Author:** Durdona Uskenbayeva
- **Advisor:** Aleksandr Kapitonov
- **Description:** Numerical solutions of Ordinary Differential Equations (ODEs), Euler vs. RK45 comparison, step-size convergence analysis, harmonic and Van der Pol oscillators.

**Learning Objectives:**
1. Understand explicit Euler integration and its limitations.
2. Compare custom numerical methods with SciPy adaptive solvers (`solve_ivp`).
3. Analyze step-size convergence and stability in differential equations.
4. Simulate second-order physical systems and nonlinear limit cycles.

## 1. Motivation (5 min)
Many physical phenomena in classical mechanics, thermodynamics, and quantum systems are naturally modeled by Ordinary Differential Equations (ODEs). While exact analytical solutions are often impossible for complex nonlinear systems, numerical integration methods allow us to simulate state trajectories, analyze stability, and predict physical behavior over time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# 1. Generalized Euler Method (Supports both scalar and vector systems)
def euler_method(f, t0, y0, h, t_end):
    t_values = np.arange(t0, t_end + h, h)
    y0 = np.atleast_1d(y0)
    y_values = np.zeros((len(t_values), len(y0)))
    y_values[0] = y0

    for i in range(1, len(t_values)):
        t = t_values[i - 1]
        y = y_values[i - 1]
        y_values[i] = y + h * np.array(f(t, y))

    return t_values, y_values

# Test differential equation: dy/dt = -2y
def dydt(t, y):
    return -2 * y

# Parameters
t0, y0, t_end, h = 0.0, 1.0, 5.0, 0.1
t_euler, y_euler = euler_method(dydt, t0, y0, h, t_end)
print("Generalized Euler method executed successfully!")

## 2. Physics background (10 min)
Governing ODE: $\frac{dy}{dt} = -2y$
Analytic solution: $y(t) = y_0 e^{-2t}$

## 3. Comparison with SciPy's RK45 solver (5 min)

In [ ]:
# Comparison with SciPy's RK45 solver
t_eval = np.linspace(t0, t_end, 100)
sol = solve_ivp(dydt, [t0, t_end], [y0], t_eval=t_eval)

plt.figure(figsize=(8, 5))
plt.plot(t_euler, y_euler, 'ro--', label='Euler Method (h=0.1)', markersize=4)
plt.plot(sol.t, sol.y[0], 'b-', label='SciPy RK45 (Exact-like)', linewidth=2)
plt.xlabel('Time (t)')
plt.ylabel('Solution y(t)')
plt.title('Comparison: Euler Method vs SciPy RK45')
plt.legend()
plt.grid(True)
plt.show()

## 4. Convergence Analysis: Impact of Step Size (h) (7 min)

In [ ]:
# Convergence Analysis: Impact of step size (h) on Euler Method Error
step_sizes = [0.2, 0.1, 0.05, 0.02, 0.01]
max_errors = []

for h_val in step_sizes:
    t_vals, y_vals = euler_method(dydt, t0, y0, h_val, t_end)
    y_exact_vals = np.exp(-2 * t_vals)
    error = np.max(np.abs(y_vals - y_exact_vals))
    max_errors.append(error)

# Plotting the error convergence graph
plt.figure(figsize=(8, 5))
plt.plot(step_sizes, max_errors, 'ro-', label='Euler Method Error', linewidth=2, markersize=6)
plt.xlabel('Step Size (h)')
plt.ylabel('Maximum Absolute Error')
plt.title('Convergence Analysis: Error vs. Step Size (h)')
plt.legend()
plt.grid(True)
plt.show()

## 5. Case Study 2: Solving Second-Order Systems (Harmonic Oscillator) (10 min)

In [ ]:
# Harmonic Oscillator system with Energy Conservation Check
def harmonic_system(t, y):
    return np.array([y[1], -y[0]])

t0_h, t_end_h = 0.0, 20.0
y0_h = np.array([1.0, 0.0]) # position = 1.0, velocity = 0.0
h_h = 0.05
t_eval_h = np.arange(t0_h, t_end_h + h_h, h_h)
sol_h = solve_ivp(harmonic_system, [t0_h, t_end_h], y0_h, t_eval=t_eval_h, method='RK45')

# Calculate Total Energy: E = 0.5 * (velocity^2 + position^2)
positions = sol_h.y[0]
velocities = sol_h.y[1]
energy = 0.5 * (velocities**2 + positions**2)

# Plotting phase space and energy conservation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Position and Velocity over time
ax1.plot(sol_h.t, positions, 'b-', label='Position y(t)', linewidth=2)
ax1.plot(sol_h.t, velocities, 'r--', label='Velocity v(t)', linewidth=1.5)
ax1.set_xlabel('Time (t)')
ax1.set_ylabel('State Variables')
ax1.set_title('Harmonic Oscillator Trajectory')
ax1.legend()
ax1.grid(True)

# 2. Energy Conservation
ax2.plot(sol_h.t, energy, 'g-', label='Total Energy E(t)', linewidth=2)
ax2.set_xlabel('Time (t)')
ax2.set_ylabel('Energy')
ax2.set_title('Energy Conservation Test ($E = \\frac{1}{2}(v^2 + y^2)$)')
ax2.set_ylim(0.45, 0.55) # Zooming in to check stability
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print(f"Initial Energy: {energy[0]:.5f}, Final Energy: {energy[-1]:.5f}")

## 6. Automated Tests (Required Verification) (5 min)

In [ ]:
# --- ENHANCED AUTOMATED TESTS ---

# 1. Test Euler method shape, length, and initial condition
t_test_e, y_test_e = euler_method(dydt, 0.0, 1.0, 0.1, 1.0)
assert len(t_test_e) == len(y_test_e), "Euler output arrays must have the same length!"
assert y_test_e[0] == 1.0, "Initial condition for Euler method is incorrect!"
assert y_test_e.ndim == 1 or y_test_e.shape[1] == 1, "Euler method should handle scalar/vector outputs properly!"

# 2. Test SciPy RK45 solver consistency using assert_allclose
sol_test = solve_ivp(dydt, [0.0, 1.0], [1.0], t_eval=[1.0])
expected_val = np.exp(-2.0 * 1.0)
np.testing.assert_allclose(sol_test.y[0][0], expected_val, rtol=1e-2,
                            err_msg="RK45 solution deviates too much from the exact value!")

# 3. Test Harmonic Oscillator shape and energy conservation bounds
assert sol_h.y.shape[0] == 2, "Harmonic oscillator must have 2 state variables (position and velocity)!"
energy_drift = np.max(np.abs(energy - energy[0]))
assert energy_drift < 0.01, "Energy conservation check failed: energy drift is too high!"

print("✅ All 5 enhanced automated tests passed successfully!")

## 7. Performance Comparison of Different Solvers (*RK45*, *Radau*, *BDF*) (5 min)

In [ ]:
# Stiff test problem (Robertson-like or stiff exponential decay)
def stiff_system(t, y):
    return np.array([-1000.0 * y[0]])

t0_stiff, t_end_stiff = 0.0, 1.0
y0_stiff = np.array([1.0])
t_eval_stiff = np.linspace(t0_stiff, t_end_stiff, 100)

# Comparing solvers on a stiff problem
sol_rk_stiff = solve_ivp(stiff_system, [t0_stiff, t_end_stiff], y0_stiff, method='RK45', t_eval=t_eval_stiff)
sol_radau = solve_ivp(stiff_system, [t0_stiff, t_end_stiff], y0_stiff, method='Radau', t_eval=t_eval_stiff)
sol_bdf = solve_ivp(stiff_system, [t0_stiff, t_end_stiff], y0_stiff, method='BDF', t_eval=t_eval_stiff)

print(f"Stiff Test Results:")
print(f"RK45 function evaluations: {sol_rk_stiff.nfev}")
print(f"Radau function evaluations: {sol_radau.nfev}")
print(f"BDF function evaluations: {sol_bdf.nfev}")

## 8. Advanced Application: Nonlinear Van der Pol Oscillator (5 min)

In [ ]:
# Van der Pol oscillator: dy0/dt = y1, dy1/dt = mu * (1 - y0**2) * y1 - y0
def vanderpol(t, y, mu=1.5):
    return np.array([y[1], mu * (1.0 - y[0]**2) * y[1] - y[0]])

t_vdp = np.linspace(0.0, 20.0, 500)
sol_vdp = solve_ivp(vanderpol, [0.0, 20.0], [2.0, 0.0], t_eval=t_vdp)

plt.figure(figsize=(9, 5))
plt.plot(sol_vdp.y[0], sol_vdp.y[1], 'm-', label='Limit Cycle ($mu=1.5$)', linewidth=2)
plt.xlabel('Position y(t)')
plt.ylabel('Velocity v(t)')
plt.title('Van der Pol Oscillator - Phase Space (Limit Cycle)')
plt.legend()
plt.grid(True)
plt.show()

## 9. Audience Exercise (10 min)
*Hands-on task cells for the audience to run and modify during the lecture.*
- **Easy Task:** Change the step size ($h$) in the Euler method from 0.1 to 0.05 and observe how the error changes on the graph.
- **Medium Task:** Complete the missing part of the harmonic oscillator step using the code cell below.
- **Stretch Task:** Try modifying the damping parameter $\mu$ in the Van der Pol oscillator and observe how the limit cycle shape changes.

## 10. Summary and questions (5 min)

In this comprehensive laboratory work, we explored the numerical solution techniques for Ordinary Differential Equations (ODEs) and their practical applications in scientific programming and physics modeling:

1. **Explicit Numerical Integration (Euler Method):** We implemented the classical first-order Euler method. While intuitive, our analysis demonstrated its conditional stability and accumulation of truncation errors over extended time horizons.
2. **Adaptive High-Order Solvers (SciPy RK45):** By comparing the Euler method with SciPy's adaptive Runge-Kutta ($RK45$) solver, we observed significant improvements in accuracy, stability, and computational efficiency.
3. **Step-Size Convergence Analysis ($h$):** The convergence study clearly illustrated the theoretical relationship between step size ($h$) and the maximum absolute error, confirming the expected error reduction rates.
4. **Second-Order Systems & Harmonic Oscillators:** We successfully modeled multi-variable physical systems by converting second-order differential equations into a system of first-order ODEs, preserving cyclical state trajectories.
5. **Advanced Stiff Solvers (Radau, BDF):** Testing alternative integration schemes highlighted the necessity of implicit solvers when dealing with complex or stiff differential equations.
6. **Nonlinear Dynamics (Van der Pol Oscillator):** The nonlinear Van der Pol oscillator simulation allowed us to visualize complex limit cycle behavior in phase space.
7. **Automated Verification:** Rigorous automated `assert` statements successfully validated code correctness, output shapes, and numerical consistency across all implemented solvers.